In [ ]:
!git clone https://github.com/Exploration-Lab/CS780-OBELIX
%cd CS780-OBELIX

!pip install -r requirements.txt
!pip install gymnasium stable-baselines3[extra]

Cloning into 'CS780-OBELIX'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 32 (delta 6), reused 4 (delta 4), pack-reused 23 (from 1)
Receiving objects: 100% (32/32), 994.04 KiB | 14.20 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/CS780-OBELIX
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 15.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

from obelix import OBELIX

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
ACTIONS = ["L45","L22","FW","R22","R45"]

class ObelixEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(0,1,(18,),dtype=np.float32)
        self.action_space = spaces.Discrete(5)
        self.prev_hit_wall = 0

    def reset(self, seed=None, options=None):

        self.env = OBELIX(
            difficulty=3,
            max_steps=1000,
            scaling_factor = 5,
            arena_size = 500,
            wall_obstacles = True
        )
        self.prev_hit_wall = 0

        obs = self.env.reset().astype(np.float32)
        return obs, {}

    def step(self, action):

        action_str = ACTIONS[action]
        obs, reward, done = self.env.step(action_str, render=False)

        obs = obs.astype(np.float32)

        # -------------------------
        # reward shaping
        # -------------------------
        front = np.sum(obs[4:12])
        side  = np.sum(obs[0:4]) + np.sum(obs[12:16])
        hit_wall = obs[-1]

        # 🔥 escape reward (FIX THIS ↓)
        # if self.prev_hit_wall == 1 and hit_wall == 0:
        #     reward += 500   # ✅ NOT 200

        # if hit_wall == 1 and action in [0,1,3,4]:
        #   reward += 200


        # update memory
        # self.prev_hit_wall = hit_wall

        # sensor shaping
        # reward += 0.05 * front
        # reward += 0.01 * side

        # -------------------------
        # 🔥 ADD MOVEMENT REWARD HERE
        # -------------------------
        if action == 2 and hit_wall == 0:  # FW
            reward += 0.7
        reward = reward/10


        return obs, reward, done, False, {}

In [ ]:
class OneLineLogger(BaseCallback):
    def __init__(self, n_envs=4, log_path="training_log.txt", log_freq=1000):
        super().__init__()
        self.n_envs = n_envs
        self.ep_rewards = [0] * n_envs
        self.completed = []

        self.log_path = log_path
        self.log_freq = log_freq

        # clear file at start
        open(self.log_path, "w").close()

    def _on_step(self):
        rewards = self.locals["rewards"]
        dones = self.locals["dones"]

        for i in range(self.n_envs):
            self.ep_rewards[i] += rewards[i]

            if dones[i]:
                self.completed.append(self.ep_rewards[i])

                avg = np.mean(self.completed[-10:])

                print(
                    f"\rSteps: {self.num_timesteps} | "
                    f"Last: {self.ep_rewards[i]:.1f} | "
                    f"Avg(10): {avg:.1f}",
                    end=""
                )

                self.ep_rewards[i] = 0

        # 🔥 LOG EVERY N STEPS
        if self.num_timesteps % self.log_freq == 0:
            avg = np.mean(self.completed[-10:]) if self.completed else 0

            with open(self.log_path, "a") as f:
                f.write(f"{self.num_timesteps},{avg}\n")

        return True

In [ ]:
from stable_baselines3.common.monitor import Monitor

def make_env():
    return Monitor(ObelixEnv())

env = DummyVecEnv([make_env for _ in range(2)])

model = PPO(
    "MlpPolicy",
    env,
    verbose=0,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.005,
)



In [ ]:
from stable_baselines3.common.callbacks import CheckpointCallback

checkpoint_callback = CheckpointCallback(
    save_freq=500_000,              # every 500K steps
    save_path="/content/drive/MyDrive/obelix_models/",
    name_prefix="ppo_obelix"
)

from stable_baselines3.common.callbacks import CallbackList
logger = OneLineLogger(n_envs=2)
callback = CallbackList([
    logger,
    checkpoint_callback
])


In [ ]:
model.learn(
    total_timesteps=1_000_000,
    callback=callback
)

In [ ]:
from stable_baselines3 import PPO
model = PPO.load(
    "/content/drive/MyDrive/obelix_models/ppo_obelix_5000000_steps.zip",
    env=env
)

In [ ]:
model = PPO.load(
    "/content/drive/MyDrive/obelix_models/ppo_obelix_1000000_steps.zip",
    env=env
)

model.learn(
    total_timesteps=1_000_000,
    callback=callback,
    reset_num_timesteps=False   # 🔥 VERY IMPORTANT
)

Steps: 1005524 | Last: -4877.3 | Avg(10): -1540.3

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/ppo/obelix_ppo_final_new18"

model.save(SAVE_PATH)
print("\n\n✅ Model saved to Drive")



✅ Model saved to Drive


In [ ]:
import torch
torch.save(model.policy.state_dict(), "/content/drive/MyDrive/ppo/ppo_weights27.pth")

In [ ]:
import torch
policy_net = model.policy.mlp_extractor.policy_net
action_net = model.policy.action_net

torch.save({
    "policy_net": policy_net.state_dict(),
    "action_net": action_net.state_dict()
}, "/content/drive/MyDrive/ppo/weights28.pth")

In [ ]:
from stable_baselines3 import PPO

model = PPO.load("/content/drive/MyDrive/ppo/obelix_ppo_final_new18", env=env)

In [ ]:
logger = OneLineLogger()
model.learn(
    total_timesteps=1_000_000,
    callback = logger,
    reset_num_timesteps=False   # 🔥 VERY IMPORTANT
)

Steps: 1123520 | Last: -899.0 | Avg(10): -3803.6

In [ ]:
print(model.policy)

ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=18, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=18, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=5, bias=True)
  (value_net): Linear(in_features=64, out_features=1, bias=True)
)


2nd Trail

In [ ]:
import os
import numpy as np
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, CallbackList
from stable_baselines3.common.monitor import Monitor

from obelix import OBELIX

# ==============================
# ACTIONS
# ==============================
ACTIONS = ["L45","L22","FW","R22","R45"]

# ==============================
# ENVIRONMENT
# ==============================
class ObelixEnv(gym.Env):

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(0,1,(18,),dtype=np.float32)
        self.action_space = spaces.Discrete(5)
        self.prev_hit_wall = 0

    def reset(self, seed=None, options=None):

        self.env = OBELIX(
            difficulty=3,
            max_steps=1000,
            scaling_factor=5,
            arena_size=500,
            wall_obstacles=True
        )

        self.prev_hit_wall = 0
        obs = self.env.reset().astype(np.float32)
        return obs, {}

    def step(self, action):

        action_str = ACTIONS[action]
        obs, reward, done = self.env.step(action_str, render=False)
        obs = obs.astype(np.float32)

        # -------------------------
        # reward shaping (balanced)
        # -------------------------
        # front = np.sum(obs[4:12])
        # side  = np.sum(obs[0:4]) + np.sum(obs[12:16])
        hit_wall = obs[-1]

        # # forward reward
        # if action == 2 and hit_wall == 0:
        #     reward += 0.5

        # # wall penalty
        # if hit_wall == 1:
        #     reward -= 2.0

        # sensor guidance
        # reward += 0.02 * front
        # reward += 0.005 * side

        # escape reward
        if self.prev_hit_wall == 1 and hit_wall == 0:
            reward += 1

        self.prev_hit_wall = hit_wall

        terminated = done
        truncated = False

        return obs, reward, terminated, truncated, {}

# ==============================
# LOGGER (RESUME SAFE)
# ==============================
class OneLineLogger(BaseCallback):
    def __init__(self, n_envs=4, log_path = "/content/drive/MyDrive/obelix_models/training_log.txt", log_freq=10000):
        super().__init__()
        self.n_envs = n_envs
        self.ep_rewards = [0] * n_envs
        self.completed = []

        self.log_path = log_path
        self.log_freq = log_freq

        # create file only if not exists
        if not os.path.exists(self.log_path):
            with open(self.log_path, "w") as f:
                f.write("steps,avg_reward\n")

    def _on_step(self):
        rewards = self.locals["rewards"]
        dones = self.locals["dones"]

        for i in range(self.n_envs):
            self.ep_rewards[i] += rewards[i]

            if dones[i]:
                self.completed.append(self.ep_rewards[i])

                avg = np.mean(self.completed[-10:])

                print(
                    f"\rSteps: {self.num_timesteps} | "
                    f"Last: {self.ep_rewards[i]:.1f} | "
                    f"Avg(10): {avg:.1f}",
                    end=""
                )

                self.ep_rewards[i] = 0

        # ---- file logging ----
        if self.num_timesteps % self.log_freq == 0:

            avg = np.mean(self.completed[-10:]) if self.completed else 0

            # avoid duplicate logging
            last_step = -1
            if os.path.exists(self.log_path):
                with open(self.log_path, "r") as f:
                    lines = f.readlines()
                    if len(lines) > 1:
                        last_step = int(lines[-1].split(",")[0])

            if self.num_timesteps > last_step:
                with open(self.log_path, "a") as f:
                    f.write(f"{self.num_timesteps},{avg}\n")

        return True

# ==============================
# ENV CREATION
# ==============================
def make_env():
    return Monitor(ObelixEnv())

env = DummyVecEnv([make_env for _ in range(4)])

# ==============================
# MODEL
# ==============================
model = PPO(
    "MlpPolicy",
    env,
    verbose=0,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.003,
)




/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
+# ==============================
# CALLBACKS
# ==============================
logger = OneLineLogger(n_envs=4)

checkpoint_callback = CheckpointCallback(
    save_freq=40_000,
    save_path="/content/drive/MyDrive/obelix_models/",
    name_prefix="ppo_obelix"
)

callback = CallbackList([logger, checkpoint_callback])
# ==============================
# TRAIN (CHANGE THIS FOR RESUME)
# ==============================
model.learn(
    total_timesteps=2_000_000,
    callback=callback
)

Steps: 2000456 | Last: -974.0 | Avg(10): -1064.9

In [ ]:


SAVE_PATH = "/content/drive/MyDrive/obelix_models/ppo_obelix_end3"

model.save(SAVE_PATH)
print("\n\n✅ Model saved to Drive")



✅ Model saved to Drive


In [ ]:
import torch
policy_net = model.policy.mlp_extractor.policy_net
action_net = model.policy.action_net

torch.save({
    "policy_net": policy_net.state_dict(),
    "action_net": action_net.state_dict()
}, "/content/drive/MyDrive/obelix_models/weights_end10.pth")

In [ ]:
import os
# ==============================
# CALLBACKS
# ==============================
logger = OneLineLogger(n_envs=4)

checkpoint_callback = CheckpointCallback(
    save_freq=40_000,
    save_path="/content/drive/MyDrive/obelix_models/",
    name_prefix="ppo_obelix"
)

callback = CallbackList([logger, checkpoint_callback])
save_path = "/content/drive/MyDrive/obelix_models/"

# files = [f for f in os.listdir(save_path) if "ppo_obelix" in f]
# latest = sorted(files, key=lambda x: int(x.split("_")[-2]))[-1]

# print("Resuming from:")

env = DummyVecEnv([make_env for _ in range(4)])

model = PPO.load(save_path + 'ppo_obelix_1760000_steps', env=env)

# model.learn(
#     total_timesteps=500_000,
#     callback=callback,
#     reset_num_timesteps=False   # 🔥 critical
# )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.loadtxt("training_log.txt", delimiter=",", skiprows=1)

steps = data[:, 0]
rewards = data[:, 1]

plt.plot(steps, rewards)
plt.xlabel("Timesteps")
plt.ylabel("Avg Reward")
plt.title("Training Curve")
plt.show()

In [ ]:
env = DummyVecEnv([make_env])

In [ ]:
env = DummyVecEnv([make_env])

In [ ]:
import torch

torch.save({
    "policy_net": model.policy.mlp_extractor.policy_net.state_dict(),
    "value_net": model.policy.mlp_extractor.value_net.state_dict(),
    "action_net": model.policy.action_net.state_dict()
}, "/content/drive/MyDrive/ppo/extracted_weights.pth")